# Qwen2.5-Coder-7B QLoRA Fine-tuning (Unsloth) — OpenCodeInstruct HQ 10K (ChatML)

Trains `Qwen/Qwen2.5-Coder-7B` with QLoRA using **Unsloth** on the high-quality 10K subset of `nvidia/OpenCodeInstruct` produced by `prepare_opencode_hq_10k.ipynb`.

**Dataset:** `chatml.jsonl` — 10,000 rows, single column `text`, already formatted as Qwen ChatML (`<|im_start|>user ... <|im_end|>\n<|im_start|>assistant ... <|im_end|>`).

In [ ]:
!pip install -q --upgrade pip
!pip install -q "unsloth==2024.11.9" "unsloth_zoo"
!pip install -q --no-deps "trl==0.12.0" "peft==0.13.2" "accelerate==1.1.1" "bitsandbytes==0.44.1"

## 1. Imports & Login

In [ ]:
import os
import glob
import json
import torch
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

try:
    wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = wandb_api_key
    REPORT_TO = "wandb"
except Exception:
    REPORT_TO = "none"

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B"
HUB_MODEL_NAME = "hoangnh39/qwen2.5-coder-7b-opencode-hq10k"

# --- Dataset source (ChatML, column "text") ---
# 1) Kaggle Dataset: file chatml.jsonl nằm đâu đó dưới /kaggle/input (tự tìm bằng glob)
# 2) Hoặc HF Hub: đặt HF_DATASET_ID = "user/repo" (bỏ qua Kaggle input)
HF_DATASET_ID = None
DATASET_FILENAME = "chatml.jsonl"

MAX_SEQ_LENGTH = 1024
MAX_TRAIN_SAMPLES = None   

## 2. Load Base Model + Attach LoRA (Unsloth)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto-detect bf16/fp16 support
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,                       # 0 is required for Unsloth's fused kernels
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth", # Unsloth's checkpointing variant, lower VRAM than vanilla
    random_state=3407,
)
print(model.print_trainable_parameters())

## 3. Load Dataset (OpenCodeInstruct HQ 10K, ChatML)

In [ ]:
if HF_DATASET_ID:
    dataset = load_dataset(HF_DATASET_ID, split="train")
else:
    matches = glob.glob(f"/kaggle/input/**/{DATASET_FILENAME}", recursive=True)
    assert matches, f"Không thấy {DATASET_FILENAME} trong /kaggle/input — hãy attach Kaggle Dataset chứa file này."
    print("Using", matches[0])
    dataset = load_dataset("json", data_files=matches[0], split="train")

assert "text" in dataset.column_names, f"Cần cột 'text', hiện có: {dataset.column_names}"
print(f"Loaded {len(dataset):,} examples")

def n_tokens(batch):
    return {"n_tok": [len(ids) for ids in tokenizer(batch["text"], add_special_tokens=False)["input_ids"]]}

dataset = dataset.map(n_tokens, batched=True, batch_size=1000)
before = len(dataset)
dataset = dataset.filter(lambda ex: ex["n_tok"] <= MAX_SEQ_LENGTH).remove_columns("n_tok")
print(f"Kept {len(dataset):,}/{before:,} samples with <= {MAX_SEQ_LENGTH} tokens")

dataset = dataset.shuffle(seed=3407)
if MAX_TRAIN_SAMPLES:
    dataset = dataset.select(range(min(MAX_TRAIN_SAMPLES, len(dataset))))
print(f"Training on {len(dataset):,} samples")
print(dataset[0]["text"][:500])

## 4. Train

In [ ]:
train_args = SFTConfig(
    output_dir="./checkpoints",
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    num_train_epochs=1,              # 10K samples: 1 epoch ~ 600 steps at eff. batch 16; tăng nếu còn quota
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,    # effective batch size 16
    optim="adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_grad_norm=0.3,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    report_to=REPORT_TO,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=train_args,
)

trainer_stats = trainer.train()
print(trainer_stats)

## 5. Save / Push Adapter

In [ ]:
model.push_to_hub(HUB_MODEL_NAME, private=True)
tokenizer.push_to_hub(HUB_MODEL_NAME, private=True)
print(f"Adapter pushed to https://huggingface.co/{HUB_MODEL_NAME}")

## 6. Smoke Test (sanity check, not a benchmark)

In [ ]:
FastLanguageModel.for_inference(model)

messages = [{"role": "user", "content": "Write a Python function `is_palindrome(s: str) -> bool` that checks whether a string is a palindrome, ignoring case and non-alphanumeric characters."}]
prompt = f"<|im_start|>user\n{messages[0]['content']}<|im_end|>\n<|im_start|>assistant\n"

im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=300, temperature=0.2, do_sample=True,
                     eos_token_id=[im_end_id, tokenizer.eos_token_id], pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))